### **Extraer el `nombre`,`año` y `población` de cada pais y exportarlo a un archivo `JSON`**

Vamos a trabajar en el enlace:

https://www.worldometers.info/world-population/population-by-country/

#### **Crear la spider**

Vamos a crear la spider:

<center><img src="https://i.postimg.cc/j25hMhhF/ws-423.png"></center>

Luego, debemos editar el archivo `worldmeters_cuatro.py` que se crea, con el código que se muestra abajo.

#### **Paso a paso**

Vamos a inspeccionar el sitio: https://www.worldometers.info/world-population/population-by-country/

<center><img src="https://i.postimg.cc/3r0KTz3w/ws-410.png"></center>

Localizamos `todas las filas` de la tabla:

<center><img src="https://i.postimg.cc/hvQx8MF0/ws-415.png"></center>

Localizamos el `nombre del pais` en cada fila:

<center><img src="https://i.postimg.cc/1t2sFrrP/ws-411.png"></center>

Localizamos el `enlace` en cada fila:

<center><img src="https://i.postimg.cc/Gmnm3DRV/ws-420.png"></center>

Ahora vamos a inspeccionar el enlace que se irá generando de manera iterativa: https://www.worldometers.info/world-population/india-population/

En dicho enlace encontramos 3 tablas en el siguiente orden:

1. La población histórica según el año
2. El pronóstico poblacional del 2025 al 2050
3. Principales ciudades según población

<center><img src="https://i.postimg.cc/G2Vr5VnR/ws-87.png"></center>
<center><img src="https://i.postimg.cc/YSXt1jXh/ws-88.png"></center>
<center><img src="https://i.postimg.cc/fydZnCM9/ws-89.png"></center>

Vamos a trabajar con la tabla poblacional histórica:

<center><img src="https://i.postimg.cc/pV3XGhPj/ws-81.png"></center>

Localizamos **`todas las filas de la primera tabla`**, de la tabla de la población histórica según año (por eso el [1]):

<center><img src="https://i.postimg.cc/3JkBpnNy/ws-424.png"></center>

Localizamos **`el año de todas las filas de la primera tabla`**, de la tabla de la población histórica según año (por eso el [1]):

<center><img src="https://i.postimg.cc/y89ypxFs/ws-425.png"></center>

Localizamos **`la población de todas las filas de la primera tabla`**, de la tabla de la población histórica según año (por eso el [1]):

<center><img src="https://i.postimg.cc/tC8zzWnB/ws-426.png"></center>

Localizamos un titulo del cual podamos extraer el `nombre del pais`

<center><img src="https://i.postimg.cc/HkygkqyB/ws-430.png"></center>

#### **Código**

##### **`Forma 1`**

Los datos extraidos por medio de este código son exportados al archivo **`worldmeters_cuatro_a.json`**

In [ ]:
import scrapy


class WorldmetersCuatroSpider(scrapy.Spider):
    name = "worldmeters_cuatro"
    allowed_domains = ["www.worldometers.info"]
    start_urls = ["https://www.worldometers.info/world-population/population-by-country"]

    def parse(self, response):
        # Extracción de nombres de paises y su población
        filas = response.xpath('//tbody/tr')

        for fila in filas:
            # Obtenemos un 'Enlace relativo' --> /world-population/india-population/
            enlace = fila.xpath('./td/a/@href').get()

            # De manera iterable, se irá enviando una petición por cada enlace que se encuentre
            # y cada página será parseada con la función 'parse_pais()'
            yield response.follow(url=enlace, callback=self.parse_pais)

    # Esta función parseará cada nuevo enlace (página) que se envie como petición:
    # Ejemplo: https://www.worldometers.info/world-population/india-population/
    #          https://www.worldometers.info/world-population/china-population/
    # etc..
    def parse_pais(self, response):
        pais = response.xpath('(//h1)[1]/text()').get(default="Sin nombre")
        filas = response.xpath('(//table[contains(@class, "table")])[1]/tbody/tr')

        for fila in filas:
            anno = fila.xpath('./td[1]/text()').get()
            poblacion = fila.xpath('./td[2]/strong/text()').get()

            # Devuelve los datos extraidos
            # Vamos a devolver un 'Enlace absoluto'
            yield {
                    'pais': pais.strip()[:-11],
                    'año':anno,
                    'poblacion': poblacion,
                }

##### **`Forma 2`**

Los datos extraidos por medio de este código son exportados al archivo **`worldmeters_cuatro_b.json`**

In [ ]:
import scrapy


class WorldmetersCuatroSpider(scrapy.Spider):
    name = "worldmeters_cuatro"
    allowed_domains = ["www.worldometers.info"]
    start_urls = ["https://www.worldometers.info/world-population/population-by-country"]

    def parse(self, response):
        # Extracción de nombres de paises y su población
        filas = response.xpath('//tbody/tr')

        for fila in filas:
            nombre_pais = fila.xpath('./td/a/text()').get()
            # Obtenemos un 'Enlace relativo' --> /world-population/india-population/
            enlace = fila.xpath('./td/a/@href').get()

            # De manera iterable, se irá enviando una petición por cada enlace que se encuentre
            # y cada página será parseada con la función 'parse_pais()'
            # Capturamos el nombre del pais y lo almacenamos en 'meta' para despues utilizarlo desde la función 'parse_pais'
            yield response.follow(url=enlace, callback=self.parse_pais, meta={'nombre_pais':nombre_pais})

    # Esta función parseará cada nuevo enlace (página) que se envie como petición:
    # Ejemplo: https://www.worldometers.info/world-population/india-population/
    #          https://www.worldometers.info/world-population/china-population/
    # etc..
    def parse_pais(self, response):
        pais = response.request.meta['nombre_pais']
        filas = response.xpath('(//table[contains(@class, "table")])[1]/tbody/tr')

        for fila in filas:
            anno = fila.xpath('./td[1]/text()').get()
            poblacion = fila.xpath('./td[2]/strong/text()').get()

            # Devuelve los datos extraidos
            # Vamos a devolver un 'Enlace absoluto'
            yield {
                    'pais': pais,
                    'año':anno,
                    'poblacion': poblacion,
                }

#### **Ejecución**

Nos posicionamos en la ruta correcta y ejecutamos nuestra Spider:

In [ ]:
scrapy crawl worldmeters_cuatro

<center><img src="https://i.postimg.cc/D0MPHbjy/ws-428.png"></center>

De esta manera obtenemos toda la info que pedimos extraer del website:

<center><img src="https://i.postimg.cc/kM1sBkJ3/ws-429.png"></center>

Podriamos exportar toda esta data en un archivo `JSON` escribiendo el siguiente comando:

In [ ]:
# Para la Forma 1
scrapy crawl worldmeters_cuatro_a -o worldmeters_cuatro_a.json

In [ ]:
# Para la Forma 2
scrapy crawl worldmeters_cuatro_b -o worldmeters_cuatro_b.json